# Task scheduling

Run both original task graphs, sample their native QAOA programs, and compare actual returned schedules with exact classical plans.

## Bind a finite scheduling polynomial at synthesis time

The visible schedule qfunc allocates the model's bounded decision width,
prepares a superposition and applies eight polynomial phase/mixer layers.
FlagQuantum compiles the declared constant and pair terms; classical problem
construction supplies task windows, capacity and penalties. Size and angles
are synthesis-time bindings, not measurement-dependent dynamic allocation.

Try a nonzero angle preview binding before altering task capacity. The small
instance retains twenty objective queries and the larger instance retains
one, both with 2048 final draws. One evaluated point is not an optimization
convergence claim. The host workflow uses its shared quantum.py builder;
propagate algorithm edits there and compare feasibility and exact classical
cost independently of sampled rank.


Load the native scheduling model and workflow.

In [ ]:
from flagquantum import synth
from flagquantum.synth import Output
from flagquantum.synth import QArray
from flagquantum.synth import QBit
from flagquantum.synth import RX
from flagquantum.synth import allocate
from flagquantum.synth import hadamard_transform
from flagquantum.synth import phase
from flagquantum.synth import qfunc

from examples.synthesis.applications.logistics.task_scheduling_problem.overview.quantum import (
    Problem,
)


@qfunc
def schedule(
    bits: int,
    count: int,
    constant: float,
    first: list[int],
    second: list[int],
    weights: list[float],
    parameters: list[float],
    target: Output[QArray[QBit]],
) -> None:
    allocate(bits, target)
    hadamard_transform(target)
    for layer in range(8):
        phase(constant, -parameters[2 * layer])
        for item in range(count):
            phase(
                weights[item] * target[first[item]] * target[second[item]],
                -parameters[2 * layer],
            )
        for bit in range(bits):
            RX(parameters[2 * layer + 1], target[bit])


def build_preview(problem: Problem, parameters: tuple[float, ...]) -> synth.Program:
    terms = [(support, weight) for support, weight in problem.polynomial if support]
    return synth.synthesize(
        schedule,
        classical_inputs={
            "bits": problem.width,
            "count": len(terms),
            "constant": dict(problem.polynomial).get((), 0.0),
            "first": [support[0] for support, _ in terms],
            "second": [support[-1] for support, _ in terms],
            "weights": [weight for _, weight in terms],
            "parameters": list(parameters),
        },
        max_qubits=problem.width,
        max_workspace=0,
        memory_limit_bytes=2**30,
    )


from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.logistics.task_scheduling_problem.overview import (
    main,
    plots,
    problem,
    quantum,
    workflow,
)


Generate the original constraints and bounded slack encoding.

In [ ]:
create = problem.create


Use the source task and resource plots.

In [ ]:
plot_workflow = plots.save


Keep both original input graph factories.

In [ ]:
small_model, large_model = create(False), create(True)


Retain the complete small graph and capacity data.

In [ ]:
assert small_model.tasks == 4 and small_model.capacity == (3, 3, 3)


Inspect its full active-variable mapping.

In [ ]:
print(small_model.active)


Retain penalty 20 and eight complete QAOA layers.

In [ ]:
small_parameters = quantum.initial_parameters(small_model)
assert len(small_parameters) == 16


Synthesize the freshly generated small model.

In [ ]:
small_program = build_preview(small_model, small_parameters)


Inspect its actual resource product.

In [ ]:
print(small_program.report["resources"])


Optimize with twenty actual CVaR queries at alpha 0.3.

In [ ]:
small = workflow.optimize(False)


Retain every measured small-model objective.

In [ ]:
assert len(small.trace) == 20


Keep all 2,048 actual final observations.

In [ ]:
assert sum(count for _, count in small.counts) == 2048


Read the selected original three-by-four matrix and cost.

In [ ]:
print(small.selected_matrix, small.selected_cost)


Retain the full observed cost distribution.

In [ ]:
assert small.counts


Keep the observed small-model schedule for plotting.

In [ ]:
small_quantum_matrix = small.selected_matrix


Solve the original small constraints by independent exact enumeration.

In [ ]:
assert problem.classical(small_model) == (small.classical_cost, small.classical_matrix)


Retain the independent classical small-model schedule.

In [ ]:
small_classical_matrix = small.classical_matrix


Retain the complete six-task, five-slot large problem.

In [ ]:
assert large_model.tasks == 6 and large_model.capacity == (1, 3, 4, 3, 1)


Inspect all fifteen active large-model assignment variables.

In [ ]:
print(large_model.active)


Generate the large model with the original penalty and eight layers.

In [ ]:
large_parameters = quantum.initial_parameters(large_model)


Compile all twenty-four original active quantum bits.

In [ ]:
large_program = build_preview(large_model, large_parameters)
print(large_program.report["resources"])


Execute the source's explicit one-query budget at alpha 0.6.

In [ ]:
large = workflow.optimize(True)


Keep all 2,048 actual final large-model draws.

In [ ]:
assert sum(count for _, count in large.counts) == 2048


Read the selected five-by-six matrix and original cost.

In [ ]:
print(large.selected_matrix, large.selected_cost)


Retain all observed large-model costs for the histogram.

In [ ]:
assert len(large.trace) == 1


Plot the actual selected schedules and both complete input models.

In [ ]:
results = (small, large)
figure_directory = Path("scheduling-figures")
plots.save(results, figure_directory)
for label in (
    "small-input",
    "small-trace",
    "small-costs",
    "small-quantum",
    "large-input",
    "large-costs",
    "large-quantum",
):
    display(Image(filename=str(figure_directory / f"{label}.png")))


Solve the original large constraints independently.

In [ ]:
assert problem.classical(large_model) == (large.classical_cost, large.classical_matrix)


Show both independent classical schedules and report every observation.

In [ ]:
for label in ("small-classical", "large-classical"):
    display(Image(filename=str(figure_directory / f"{label}.png")))
main.report(results)
main.write(results, Path("scheduling-results.json"))
